# Taller 8: Valores Influyentes y Robustez

En este taller exploraremos los conceptos de valores influyentes (outliers e influyentes) en modelos lineales y las técnicas de regresión robusta. Analizaremos cómo identificar observaciones que tienen una influencia desproporcionada en el modelo y cómo aplicar métodos robustos para mitigar su impacto.

## Objetivos de Aprendizaje
- Implementar métricas de influencia desde cero (leverage, DFBETAS, DFFITS, Distancia de Cook)
- Comprender los fundamentos teóricos de estimadores robustos
- Evaluar el impacto de valores atípicos en modelos de regresión
- Implementar y comparar diferentes métodos de regresión robusta


In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.linalg import inv
from sklearn.linear_model import LinearRegression, RANSACRegressor, HuberRegressor
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.graphics.influence_plots import influence_plot
from statsmodels.robust.robust_linear_model import RLM
from statsmodels.tools.tools import add_constant
from scipy.stats import t

# Configuración para visualizaciones
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')
%matplotlib inline

# Configurar opciones de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
np.set_printoptions(precision=4, suppress=True)


# Teoría: Valores Influyentes y Robustez

## Conceptos Fundamentales

### Valores Atípicos vs. Valores Influyentes

Es importante distinguir entre estos dos conceptos:

- **Valores atípicos (outliers)**: Observaciones con valores que se desvían considerablemente del patrón general de los datos, ya sea en la variable respuesta (outliers en $y$) o en las variables predictoras (outliers en $X$).

- **Valores influyentes**: Observaciones que tienen un impacto desproporcionado en los parámetros estimados del modelo. Un valor puede ser atípico pero no necesariamente influyente, y viceversa.

### ¿Por qué nos preocupan los valores influyentes?

Los valores influyentes pueden:

1. Distorsionar significativamente los coeficientes estimados
2. Afectar las conclusiones estadísticas derivadas del modelo
3. Reducir la capacidad predictiva para observaciones futuras
4. Violar los supuestos del modelo lineal clásico

## Medidas de Influencia

### 1. Leverage (Apalancamiento)

El leverage mide la influencia potencial de una observación basándose únicamente en sus valores en las variables predictoras (no considera la variable respuesta).

Para la observación $i$, el leverage $h_i$ es el $i$-ésimo elemento diagonal de la matriz "hat":

$$H = X(X'X)^{-1}X'$$

donde $X$ es la matriz de diseño incluyendo la columna de unos para el intercepto.

**Interpretación**:
- $h_i$ toma valores entre $\frac{1}{n}$ y 1
- Valores altos de $h_i$ indican observaciones con combinaciones inusuales o extremas de valores en las variables predictoras
- Punto de corte común: $h_i > 2(p+1)/n$ donde $p$ es el número de predictores y $n$ el número de observaciones

### 2. Residuos Estandarizados y Estudentizados

Los residuos pueden estandarizarse para facilitar su interpretación:

**Residuos estandarizados**:
$$r_i = \frac{e_i}{\hat{\sigma}\sqrt{1-h_i}}$$

**Residuos estudentizados internos**:
$$t_i = \frac{e_i}{\hat{\sigma}\sqrt{1-h_i}}$$

**Residuos estudentizados externos (jackknife)**:
$$t_i^* = \frac{e_i}{\hat{\sigma}_{(i)}\sqrt{1-h_i}}$$

donde $\hat{\sigma}_{(i)}$ es la estimación de $\sigma$ omitiendo la observación $i$.

### 3. DFFITS

DFFITS mide cuánto cambia el valor ajustado $\hat{y}_i$ cuando la observación $i$ se excluye del modelo:

$$\text{DFFITS}_i = \frac{\hat{y}_i - \hat{y}_{i(i)}}{\hat{\sigma}_{(i)}\sqrt{h_i}} = t_i^* \sqrt{\frac{h_i}{1-h_i}}$$

donde $\hat{y}_{i(i)}$ es el valor ajustado para la observación $i$ cuando se omite del ajuste del modelo.

**Punto de corte común**: $|\text{DFFITS}_i| > 2\sqrt{\frac{p+1}{n}}$

### 4. DFBETAS

DFBETAS mide el cambio en los coeficientes estimados cuando se excluye la observación $i$:

$$\text{DFBETAS}_{ij} = \frac{\hat{\beta}_j - \hat{\beta}_{j(i)}}{\hat{\sigma}_{(i)}\sqrt{(X'X)^{-1}_{jj}}}$$

donde $\hat{\beta}_{j(i)}$ es el estimador del coeficiente $j$ cuando se excluye la observación $i$.

**Punto de corte común**: $|\text{DFBETAS}_{ij}| > \frac{2}{\sqrt{n}}$

### 5. Distancia de Cook

La distancia de Cook es una medida integral que combina cambios en los coeficientes y en los valores ajustados:

$$D_i = \frac{\sum_{j=1}^n (\hat{y}_j - \hat{y}_{j(i)})^2}{(p+1)\hat{\sigma}^2} = \frac{e_i^2}{(p+1)\hat{\sigma}^2} \cdot \frac{h_i}{(1-h_i)^2}$$

**Punto de corte común**: $D_i > \frac{4}{n-p-1}$ o $D_i > 1$

## Regresión Robusta

La regresión robusta ofrece alternativas al método de mínimos cuadrados ordinarios (OLS) que son menos sensibles a valores atípicos.

### Métodos comunes de regresión robusta:

1. **M-estimadores**: Minimizan una función de pérdida robusta en lugar de la suma de errores cuadráticos.

2. **MM-estimadores**: Combinan alta eficiencia y un alto punto de ruptura.

3. **S-estimadores**: Minimizan una estimación robusta de la escala de los residuos.

4. **Least Trimmed Squares (LTS)**: Minimiza la suma de los $h$ residuos cuadráticos más pequeños.

5. **RANSAC (Random Sample Consensus)**: Identifica y excluye outliers mediante un proceso iterativo.

### Funciones de pérdida robustas

Los M-estimadores utilizan funciones de pérdida alternativas, como:

1. **Huber**:
   $$\rho(e) = \begin{cases}
   \frac{1}{2}e^2 & \text{si } |e| \leq k \\
   k|e| - \frac{1}{2}k^2 & \text{si } |e| > k
   \end{cases}$$

2. **Tukey's Biweight**:
   $$\rho(e) = \begin{cases}
   \frac{k^2}{6}\left[1-\left(1-\left(\frac{e}{k}\right)^2\right)^3\right] & \text{si } |e| \leq k \\
   \frac{k^2}{6} & \text{si } |e| > k
   \end{cases}$$

3. **Andrews Wave**:
   $$\rho(e) = \begin{cases}
   k\left(1-\cos\left(\frac{e}{k}\right)\right) & \text{si } |e| \leq k\pi \\
   2k & \text{si } |e| > k\pi
   \end{cases}$$

Estas funciones asignan menos peso a los residuos grandes, reduciendo la influencia de los valores atípicos.


# Implementación de Métricas de Influencia

A continuación, implementaremos desde cero las principales métricas para detectar valores influyentes en modelos de regresión lineal.


In [ ]:

def calcular_hat_matrix(X):
    """
    Calcula la matriz hat (o matriz de proyección).

    Parámetros:
    -----------
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.

    Retorna:
    --------
    H : ndarray
        Matriz hat.
    """
    # Convertir a matriz numpy si no lo es
    X = np.asarray(X)

    # Calcular la matriz hat: H = X(X'X)^{-1}X'
    H = X @ np.linalg.inv(X.T @ X) @ X.T

    return H

def calcular_leverage(X):
    """
    Calcula los valores de leverage (diagonal de la matriz hat).

    Parámetros:
    -----------
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.

    Retorna:
    --------
    h : ndarray
        Vector de valores de leverage.
    """
    # Calcular la matriz hat
    H = calcular_hat_matrix(X)

    # Extraer la diagonal (valores de leverage)
    h = np.diag(H)

    return h

def calcular_residuos_estandarizados(modelo, X, y):
    """
    Calcula los residuos estandarizados.

    Parámetros:
    -----------
    modelo : objeto del modelo ajustado
        Modelo de regresión ajustado.
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.
    y : array-like
        Variable respuesta.

    Retorna:
    --------
    residuos_estandarizados : ndarray
        Vector de residuos estandarizados.
    """
    # Obtener residuos
    y_pred = modelo.predict(X)
    residuos = y - y_pred

    # Calcular valores de leverage
    h = calcular_leverage(X)

    # Estimar sigma^2
    n, p = X.shape
    sigma2 = np.sum(residuos**2) / (n - p)

    # Calcular residuos estandarizados
    residuos_estandarizados = residuos / np.sqrt(sigma2 * (1 - h))

    return residuos_estandarizados

def calcular_residuos_estudentizados(modelo, X, y):
    """
    Calcula los residuos estudentizados internos.

    Parámetros:
    -----------
    modelo : objeto del modelo ajustado
        Modelo de regresión ajustado.
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.
    y : array-like
        Variable respuesta.

    Retorna:
    --------
    residuos_estudentizados : ndarray
        Vector de residuos estudentizados internos.
    """
    # Es equivalente a los residuos estandarizados
    return calcular_residuos_estandarizados(modelo, X, y)

def calcular_residuos_estudentizados_externos(modelo, X, y):
    """
    Calcula los residuos estudentizados externos (jackknife).

    Parámetros:
    -----------
    modelo : objeto del modelo ajustado
        Modelo de regresión ajustado.
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.
    y : array-like
        Variable respuesta.

    Retorna:
    --------
    residuos_estudentizados_externos : ndarray
        Vector de residuos estudentizados externos.
    """
    # Obtener dimensiones
    n, p = X.shape

    # Obtener residuos y leverage
    y_pred = modelo.predict(X)
    residuos = y - y_pred
    h = calcular_leverage(X)

    # Calcular sigma^2
    sigma2 = np.sum(residuos**2) / (n - p)

    # Inicializar array para residuos estudentizados externos
    residuos_estudentizados_externos = np.zeros(n)

    # Calcular residuos estudentizados externos para cada observación
    for i in range(n):
        # Calcular sigma^2 omitiendo la observación i
        sigma2_i = ((n - p) * sigma2 - residuos[i]**2 / (1 - h[i])) / (n - p - 1)

        # Calcular residuo estudentizado externo
        residuos_estudentizados_externos[i] = residuos[i] / np.sqrt(sigma2_i * (1 - h[i]))

    return residuos_estudentizados_externos

def calcular_dffits(modelo, X, y):
    """
    Calcula los valores DFFITS.

    Parámetros:
    -----------
    modelo : objeto del modelo ajustado
        Modelo de regresión ajustado.
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.
    y : array-like
        Variable respuesta.

    Retorna:
    --------
    dffits : ndarray
        Vector de valores DFFITS.
    """
    # Obtener residuos estudentizados externos
    residuos_estudentizados_externos = calcular_residuos_estudentizados_externos(modelo, X, y)

    # Obtener valores de leverage
    h = calcular_leverage(X)

    # Calcular DFFITS
    dffits = residuos_estudentizados_externos * np.sqrt(h / (1 - h))

    return dffits

def calcular_dfbetas(modelo, X, y):
    """
    Calcula los valores DFBETAS.

    Parámetros:
    -----------
    modelo : objeto del modelo ajustado
        Modelo de regresión ajustado.
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.
    y : array-like
        Variable respuesta.

    Retorna:
    --------
    dfbetas : ndarray
        Matriz de valores DFBETAS (filas: observaciones, columnas: coeficientes).
    """
    # Obtener dimensiones
    n, p = X.shape

    # Obtener residuos y leverage
    y_pred = modelo.predict(X)
    residuos = y - y_pred
    h = calcular_leverage(X)

    # Calcular sigma^2
    sigma2 = np.sum(residuos**2) / (n - p)

    # Calcular (X'X)^{-1}X'
    XtXinvXt = np.linalg.inv(X.T @ X) @ X.T

    # Inicializar matriz para DFBETAS
    dfbetas = np.zeros((n, p))

    # Calcular DFBETAS para cada observación
    for i in range(n):
        # Calcular sigma^2 omitiendo la observación i
        sigma2_i = ((n - p) * sigma2 - residuos[i]**2 / (1 - h[i])) / (n - p - 1)
        sigma_i = np.sqrt(sigma2_i)

        # Calcular DFBETAS para cada coeficiente
        for j in range(p):
            dfbetas[i, j] = residuos[i] * XtXinvXt[j, i] / (sigma_i * np.sqrt(1 - h[i]) * np.sqrt(np.linalg.inv(X.T @ X)[j, j]))

    return dfbetas

def calcular_distancia_cook(modelo, X, y):
    """
    Calcula la distancia de Cook.

    Parámetros:
    -----------
    modelo : objeto del modelo ajustado
        Modelo de regresión ajustado.
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.
    y : array-like
        Variable respuesta.

    Retorna:
    --------
    D : ndarray
        Vector de distancias de Cook.
    """
    # Obtener dimensiones
    n, p = X.shape

    # Obtener residuos y leverage
    y_pred = modelo.predict(X)
    residuos = y - y_pred
    h = calcular_leverage(X)

    # Calcular sigma^2
    sigma2 = np.sum(residuos**2) / (n - p)

    # Calcular residuos estandarizados
    residuos_estandarizados = residuos / np.sqrt(sigma2 * (1 - h))

    # Calcular distancia de Cook
    D = (residuos_estandarizados**2 / p) * (h / (1 - h))

    return D

def identificar_valores_influyentes(modelo, X, y, nombres=None):
    """
    Identifica valores potencialmente influyentes según diferentes métricas.

    Parámetros:
    -----------
    modelo : objeto del modelo ajustado
        Modelo de regresión ajustado.
    X : array-like
        Matriz de diseño, incluyendo columna de unos para el intercepto.
    y : array-like
        Variable respuesta.
    nombres : array-like, opcional
        Nombres o índices de las observaciones.

    Retorna:
    --------
    resultados : DataFrame
        DataFrame con las métricas de influencia y flags para valores influyentes.
    """
    # Convertir a arrays numpy
    X = np.asarray(X)
    y = np.asarray(y)

    # Obtener dimensiones
    n, p = X.shape

    # Calcular métricas de influencia
    h = calcular_leverage(X)
    residuos_estandarizados = calcular_residuos_estandarizados(modelo, X, y)
    dffits = calcular_dffits(modelo, X, y)
    distancia_cook = calcular_distancia_cook(modelo, X, y)

    # Definir puntos de corte
    cutoff_leverage = 2 * p / n
    cutoff_residuos = 2  # Aproximadamente 2 desviaciones estándar
    cutoff_dffits = 2 * np.sqrt(p / n)
    cutoff_cook = 4 / (n - p)

    # Crear DataFrame con resultados
    if nombres is None:
        nombres = np.arange(n)

    resultados = pd.DataFrame({
        'Observación': nombres,
        'Leverage': h,
        'Residuos Estandarizados': residuos_estandarizados,
        'DFFITS': dffits,
        'Distancia de Cook': distancia_cook,
        'Flag Leverage': h > cutoff_leverage,
        'Flag Residuos': np.abs(residuos_estandarizados) > cutoff_residuos,
        'Flag DFFITS': np.abs(dffits) > cutoff_dffits,
        'Flag Cook': distancia_cook > cutoff_cook
    })

    # Añadir columna de resumen
    resultados['Total Flags'] = resultados[['Flag Leverage', 'Flag Residuos', 'Flag DFFITS', 'Flag Cook']].sum(axis=1)

    return resultados


# Implementación de Métodos de Regresión Robusta

A continuación, implementaremos varios métodos de regresión robusta desde cero, incluyendo M-estimadores con diferentes funciones de pérdida.


In [ ]:

def funcion_peso_huber(residuos, k=1.345):
    """
    Función de peso de Huber para M-estimación.

    Parámetros:
    -----------
    residuos : array-like
        Vector de residuos.
    k : float, opcional
        Parámetro de ajuste (default: 1.345 para 95% de eficiencia en distribución normal).

    Retorna:
    --------
    pesos : ndarray
        Vector de pesos para cada observación.
    """
    # Convertir a array numpy
    residuos = np.asarray(residuos)

    # Calcular pesos según la función de Huber
    pesos = np.ones_like(residuos)
    mask = np.abs(residuos) > k
    pesos[mask] = k / np.abs(residuos[mask])

    return pesos

def funcion_peso_tukey(residuos, k=4.685):
    """
    Función de peso de Tukey (Biweight) para M-estimación.

    Parámetros:
    -----------
    residuos : array-like
        Vector de residuos.
    k : float, opcional
        Parámetro de ajuste (default: 4.685 para 95% de eficiencia en distribución normal).

    Retorna:
    --------
    pesos : ndarray
        Vector de pesos para cada observación.
    """
    # Convertir a array numpy
    residuos = np.asarray(residuos)

    # Calcular pesos según la función de Tukey
    r_abs = np.abs(residuos)
    pesos = np.zeros_like(residuos)
    mask = r_abs <= k
    pesos[mask] = (1 - (residuos[mask]/k)**2)**2

    return pesos

def regresion_robusta_iterativa(X, y, func_peso=funcion_peso_huber, k=1.345, max_iter=100, tol=1e-5):
    """
    Implementa regresión robusta mediante iteratively reweighted least squares (IRLS).

    Parámetros:
    -----------
    X : array-like
        Matriz de diseño, sin incluir columna de unos para el intercepto.
    y : array-like
        Variable respuesta.
    func_peso : función, opcional
        Función de peso para la M-estimación (default: funcion_peso_huber).
    k : float, opcional
        Parámetro de ajuste para la función de peso.
    max_iter : int, opcional
        Número máximo de iteraciones (default: 100).
    tol : float, opcional
        Tolerancia para convergencia (default: 1e-5).

    Retorna:
    --------
    coef : ndarray
        Vector de coeficientes estimados (incluyendo intercepto).
    historial_coef : list
        Lista de coeficientes en cada iteración.
    convergencia : bool
        True si el algoritmo convergió, False en caso contrario.
    """
    # Añadir columna de unos para el intercepto
    X = np.asarray(X)
    y = np.asarray(y)
    X_con_intercepto = add_constant(X)

    # Inicializar con OLS
    coef = np.linalg.inv(X_con_intercepto.T @ X_con_intercepto) @ X_con_intercepto.T @ y

    # Para almacenar historial de coeficientes
    historial_coef = [coef.copy()]

    # Iteraciones de IRLS
    convergencia = False
    for _ in range(max_iter):
        # Calcular residuos
        y_pred = X_con_intercepto @ coef
        residuos = y - y_pred

        # Estimar escala robusta (MAD)
        escala = np.median(np.abs(residuos - np.median(residuos))) / 0.6745

        # Residuos estandarizados
        if escala < np.finfo(float).eps:
            escala = 1.0  # Evitar división por cero
        u = residuos / escala

        # Calcular pesos
        pesos = func_peso(u, k)

        # Ajustar modelo ponderado
        W = np.diag(pesos)
        XtWX = X_con_intercepto.T @ W @ X_con_intercepto
        XtWy = X_con_intercepto.T @ W @ y
        coef_nuevo = np.linalg.inv(XtWX) @ XtWy

        # Verificar convergencia
        delta = np.linalg.norm(coef_nuevo - coef) / np.linalg.norm(coef)
        historial_coef.append(coef_nuevo.copy())

        if delta < tol:
            convergencia = True
            coef = coef_nuevo
            break

        coef = coef_nuevo

    return coef, historial_coef, convergencia

def regresion_lts(X, y, h=None, n_submuestras=1000):
    """
    Implementa Least Trimmed Squares (LTS).

    Parámetros:
    -----------
    X : array-like
        Matriz de diseño, sin incluir columna de unos para el intercepto.
    y : array-like
        Variable respuesta.
    h : int, opcional
        Número de observaciones a utilizar (default: floor((n+p+1)/2)).
    n_submuestras : int, opcional
        Número de submuestras aleatorias a considerar (default: 1000).

    Retorna:
    --------
    coef : ndarray
        Vector de coeficientes estimados (incluyendo intercepto).
    subset : ndarray
        Índices de las observaciones utilizadas en la estimación final.
    """
    # Añadir columna de unos para el intercepto
    X = np.asarray(X)
    y = np.asarray(y)
    X_con_intercepto = add_constant(X)

    # Obtener dimensiones
    n, p = X_con_intercepto.shape

    # Calcular h si no se proporciona
    if h is None:
        h = int(np.floor((n + p + 1) / 2))

    # Validar que h es un valor válido
    if h < p:
        raise ValueError("h debe ser al menos igual al número de parámetros")
    if h > n:
        raise ValueError("h no puede ser mayor que el número de observaciones")

    # Inicializar mejor solución
    mejor_suma_residuos2 = np.inf
    mejor_coef = None
    mejor_subset = None

    # Generar submuestras aleatorias
    for _ in range(n_submuestras):
        # Seleccionar muestra aleatoria de p observaciones
        indices_submuestra = np.random.choice(n, size=p, replace=False)

        try:
            # Ajustar modelo en la submuestra
            X_sub = X_con_intercepto[indices_submuestra, :]
            y_sub = y[indices_submuestra]
            coef = np.linalg.inv(X_sub.T @ X_sub) @ X_sub.T @ y_sub

            # Calcular residuos al cuadrado para todas las observaciones
            residuos = y - X_con_intercepto @ coef
            residuos2 = residuos**2

            # Ordenar los residuos al cuadrado
            indices_ordenados = np.argsort(residuos2)

            # Calcular la suma de los h residuos al cuadrado más pequeños
            suma_residuos2 = np.sum(residuos2[indices_ordenados[:h]])

            # Actualizar mejor solución si es necesario
            if suma_residuos2 < mejor_suma_residuos2:
                mejor_suma_residuos2 = suma_residuos2
                mejor_subset = indices_ordenados[:h]

                # Recalcular coeficientes con los h mejores puntos
                X_mejor = X_con_intercepto[mejor_subset, :]
                y_mejor = y[mejor_subset]
                mejor_coef = np.linalg.inv(X_mejor.T @ X_mejor) @ X_mejor.T @ y_mejor
        except np.linalg.LinAlgError:
            # Ignorar submuestras singulares
            continue

    if mejor_coef is None:
        raise ValueError("No se pudo encontrar una solución válida")

    return mejor_coef, mejor_subset

def comparar_metodos_robustos(X, y, outliers_indices=None):
    """
    Compara diferentes métodos de regresión robusta.

    Parámetros:
    -----------
    X : array-like
        Matriz de diseño, sin incluir columna de unos para el intercepto.
    y : array-like
        Variable respuesta.
    outliers_indices : array-like, opcional
        Índices de outliers conocidos para visualización.

    Retorna:
    --------
    resultados : DataFrame
        DataFrame con los coeficientes estimados por cada método.
    """
    # Convertir a arrays numpy
    X = np.asarray(X)
    y = np.asarray(y)

    # Añadir columna de intercepto para nuestras implementaciones
    X_con_intercepto = add_constant(X)

    # Ajustar modelos
    modelos = {}

    # OLS clásico
    ols_coef = np.linalg.inv(X_con_intercepto.T @ X_con_intercepto) @ X_con_intercepto.T @ y
    modelos['OLS'] = ols_coef

    # M-estimadores con diferentes funciones de peso
    huber_coef, _, _ = regresion_robusta_iterativa(X, y, func_peso=funcion_peso_huber)
    modelos['Huber'] = huber_coef

    tukey_coef, _, _ = regresion_robusta_iterativa(X, y, func_peso=funcion_peso_tukey)
    modelos['Tukey'] = tukey_coef

    # LTS
    try:
        lts_coef, _ = regresion_lts(X, y, n_submuestras=100)  # Reducir número para velocidad
        modelos['LTS'] = lts_coef
    except Exception as e:
        print(f"Error en LTS: {e}")

    # Statsmodels para comparar
    sm_X = add_constant(X)

    # RLM con Huber
    try:
        sm_huber = sm.RLM(y, sm_X, M=sm.robust.norms.HuberT()).fit()
        modelos['SM Huber'] = sm_huber.params
    except Exception as e:
        print(f"Error en SM Huber: {e}")

    # RLM con Tukey's Biweight
    try:
        sm_tukey = sm.RLM(y, sm_X, M=sm.robust.norms.TukeyBiweight()).fit()
        modelos['SM Tukey'] = sm_tukey.params
    except Exception as e:
        print(f"Error en SM Tukey: {e}")

    # RANSAC de sklearn
    try:
        ransac = RANSACRegressor(random_state=42)
        ransac.fit(X, y)
        ransac_coef = np.concatenate(([ransac.estimator_.intercept_], ransac.estimator_.coef_))
        modelos['RANSAC'] = ransac_coef
    except Exception as e:
        print(f"Error en RANSAC: {e}")

    # Huber de sklearn
    try:
        huber_sklearn = HuberRegressor(epsilon=1.35)
        huber_sklearn.fit(X, y)
        huber_sklearn_coef = np.concatenate(([huber_sklearn.intercept_], huber_sklearn.coef_))
        modelos['SK Huber'] = huber_sklearn_coef
    except Exception as e:
        print(f"Error en SK Huber: {e}")

    # Crear DataFrame con resultados
    coef_names = ['Intercepto'] + [f'Beta_{i+1}' for i in range(X.shape[1])]
    resultados = pd.DataFrame(modelos, index=coef_names).T

    return resultados


# Generación de datos para ejemplos

A continuación, generaremos datos sintéticos para ilustrar los conceptos de valores influyentes y métodos robustos, incluyendo ejemplos con y sin outliers.


In [ ]:

def generar_datos_lineales(n=100, p=2, beta_true=None, sigma=1.0, seed=42, outliers_prop=0.0, outliers_factor=5):
    """
    Genera datos sintéticos para regresión lineal, con opción de incluir outliers.

    Parámetros:
    -----------
    n : int
        Número de observaciones
    p : int
        Número de predictores
    beta_true : array-like, opcional
        Coeficientes reales [intercepto, beta_1, beta_2, ...]
    sigma : float
        Desviación estándar del error
    seed : int
        Semilla para reproducibilidad
    outliers_prop : float
        Proporción de outliers a incluir (0 a 1)
    outliers_factor : float
        Factor de multiplicación para los outliers

    Retorna:
    --------
    X : DataFrame
        Variables predictoras
    y : Series
        Variable respuesta
    outliers_indices : ndarray
        Índices de los outliers generados
    """
    np.random.seed(seed)

    # Generar coeficientes reales si no se proporcionan
    if beta_true is None:
        beta_true = np.concatenate(([2.0], np.random.uniform(-3, 3, p)))

    # Generar predictores (variables normales)
    X = np.random.normal(0, 1, size=(n, p))

    # Convertir a DataFrame para mejor manejo
    X_df = pd.DataFrame(X, columns=[f'X{i+1}' for i in range(p)])

    # Generar variable respuesta
    intercepto = beta_true[0]
    error = np.random.normal(0, sigma, n)

    y = intercepto + X @ beta_true[1:] + error
    y_series = pd.Series(y, name='Y')

    # Añadir outliers
    n_outliers = int(n * outliers_prop)
    outliers_indices = np.random.choice(n, n_outliers, replace=False)

    # Outliers en X (apalancamiento)
    for i in outliers_indices:
        X_df.iloc[i, :] = X_df.iloc[i, :] * outliers_factor

    # Outliers en Y (respuesta)
    y_series.iloc[outliers_indices] = y_series.iloc[outliers_indices] + np.random.choice([-1, 1], n_outliers) * outliers_factor * sigma

    return X_df, y_series, outliers_indices

# Generar conjunto de datos sin outliers
X_clean, y_clean, _ = generar_datos_lineales(n=100, p=2, outliers_prop=0.0, seed=42)

# Generar conjunto de datos con outliers
X_outliers, y_outliers, outliers_indices = generar_datos_lineales(n=100, p=2, outliers_prop=0.1, outliers_factor=7, seed=42)

# Mostrar estadísticas descriptivas
print("Estadísticas descriptivas de los datos sin outliers:")
print(pd.concat([X_clean, y_clean], axis=1).describe())

print("\nEstadísticas descriptivas de los datos con outliers:")
print(pd.concat([X_outliers, y_outliers], axis=1).describe())

# Visualizar los datos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Datos sin outliers
axes[0, 0].scatter(X_clean['X1'], y_clean)
axes[0, 0].set_title('Datos sin outliers: X1 vs Y')
axes[0, 0].set_xlabel('X1')
axes[0, 0].set_ylabel('Y')

axes[0, 1].scatter(X_clean['X2'], y_clean)
axes[0, 1].set_title('Datos sin outliers: X2 vs Y')
axes[0, 1].set_xlabel('X2')
axes[0, 1].set_ylabel('Y')

# Datos con outliers
axes[1, 0].scatter(X_outliers['X1'], y_outliers)
axes[1, 0].scatter(X_outliers.iloc[outliers_indices, 0], y_outliers.iloc[outliers_indices], 
                  color='red', marker='x', s=100, label='Outliers')
axes[1, 0].set_title('Datos con outliers: X1 vs Y')
axes[1, 0].set_xlabel('X1')
axes[1, 0].set_ylabel('Y')
axes[1, 0].legend()

axes[1, 1].scatter(X_outliers['X2'], y_outliers)
axes[1, 1].scatter(X_outliers.iloc[outliers_indices, 1], y_outliers.iloc[outliers_indices], 
                  color='red', marker='x', s=100, label='Outliers')
axes[1, 1].set_title('Datos con outliers: X2 vs Y')
axes[1, 1].set_xlabel('X2')
axes[1, 1].set_ylabel('Y')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Crear una matriz de dispersión para visualizar relaciones
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Para la visualización pairplot, necesitamos hacer un ajuste
# ya que estamos usando una versión más moderna de seaborn
sns.pairplot(pd.concat([X_clean, y_clean], axis=1))
plt.suptitle('Matriz de dispersión - Datos sin outliers', y=1.02)
plt.tight_layout()
plt.show()

# Crear un DataFrame con una columna que identifique outliers
df_outliers = pd.concat([X_outliers, y_outliers], axis=1)
df_outliers['outlier'] = 0
df_outliers.loc[outliers_indices, 'outlier'] = 1

sns.pairplot(df_outliers, hue='outlier')
plt.suptitle('Matriz de dispersión - Datos con outliers', y=1.02)
plt.tight_layout()
plt.show()


# Análisis de Valores Influyentes

En esta sección, aplicaremos las métricas implementadas para detectar valores influyentes en nuestros datos.


In [ ]:

# Función auxiliar para ajustar modelos
def ajustar_modelo_ols(X, y):
    """Ajusta un modelo OLS y devuelve el objeto de resultados."""
    X_con_intercepto = add_constant(X)
    return sm.OLS(y, X_con_intercepto).fit()

# Ajustar modelos a los datos sin y con outliers
modelo_clean = ajustar_modelo_ols(X_clean, y_clean)
modelo_outliers = ajustar_modelo_ols(X_outliers, y_outliers)

print("Resumen del modelo sin outliers:")
print(modelo_clean.summary())

print("\nResumen del modelo con outliers:")
print(modelo_outliers.summary())

# Comparación de coeficientes
coef_clean = modelo_clean.params
coef_outliers = modelo_outliers.params

print("\nComparación de coeficientes:")
print(f"{'Parámetro':<10} {'Sin outliers':<15} {'Con outliers':<15} {'Diferencia %':<15}")
print("-" * 55)
for i, name in enumerate(['Intercepto', 'X1', 'X2']):
    diff_pct = 100 * abs(coef_clean[i] - coef_outliers[i]) / abs(coef_clean[i])
    print(f"{name:<10} {coef_clean[i]:<15.4f} {coef_outliers[i]:<15.4f} {diff_pct:<15.2f}")

# Identificar valores influyentes en el conjunto sin outliers
print("\nAnálisis de valores influyentes en datos sin outliers:")
X_clean_const = add_constant(X_clean)
influyentes_clean = identificar_valores_influyentes(modelo_clean, X_clean_const, y_clean)
print(influyentes_clean.sort_values('Total Flags', ascending=False).head(10))

# Identificar valores influyentes en el conjunto con outliers
print("\nAnálisis de valores influyentes en datos con outliers:")
X_outliers_const = add_constant(X_outliers)
influyentes_outliers = identificar_valores_influyentes(modelo_outliers, X_outliers_const, y_outliers)
print(influyentes_outliers.sort_values('Total Flags', ascending=False).head(10))

# Comparar con outliers conocidos
outliers_detectados = set(influyentes_outliers[influyentes_outliers['Total Flags'] >= 2].index)
outliers_reales = set(outliers_indices)
print("\nEfectividad de la detección de outliers:")
print(f"Outliers reales: {len(outliers_reales)}")
print(f"Outliers detectados (≥2 flags): {len(outliers_detectados)}")
print(f"Verdaderos positivos: {len(outliers_reales.intersection(outliers_detectados))}")
print(f"Falsos positivos: {len(outliers_detectados - outliers_reales)}")
print(f"Falsos negativos: {len(outliers_reales - outliers_detectados)}")

# Visualización de métricas de influencia
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Gráfico de leverage vs residuos estudentizados
axes[0, 0].scatter(influyentes_outliers['Leverage'], influyentes_outliers['Residuos Estandarizados'])
axes[0, 0].set_xlabel('Leverage (h)')
axes[0, 0].set_ylabel('Residuos Estandarizados')
axes[0, 0].set_title('Leverage vs Residuos Estandarizados')
# Añadir líneas de referencia
cutoff_leverage = 2 * (X_outliers.shape[1] + 1) / X_outliers.shape[0]
axes[0, 0].axhline(y=2, color='r', linestyle='--')
axes[0, 0].axhline(y=-2, color='r', linestyle='--')
axes[0, 0].axvline(x=cutoff_leverage, color='r', linestyle='--')
# Marcar outliers conocidos
axes[0, 0].scatter(influyentes_outliers.iloc[outliers_indices]['Leverage'], 
                  influyentes_outliers.iloc[outliers_indices]['Residuos Estandarizados'],
                  color='red', marker='x', s=100)

# Gráfico de distancia de Cook
axes[0, 1].stem(influyentes_outliers.index, influyentes_outliers['Distancia de Cook'])
axes[0, 1].set_xlabel('Índice de observación')
axes[0, 1].set_ylabel('Distancia de Cook')
axes[0, 1].set_title('Distancia de Cook por observación')
# Añadir línea de referencia
cutoff_cook = 4 / (X_outliers.shape[0] - X_outliers.shape[1] - 1)
axes[0, 1].axhline(y=cutoff_cook, color='r', linestyle='--')
# Marcar outliers conocidos
axes[0, 1].scatter(outliers_indices, influyentes_outliers.iloc[outliers_indices]['Distancia de Cook'],
                  color='red', marker='x', s=100)

# Gráfico de DFFITS
axes[1, 0].stem(influyentes_outliers.index, influyentes_outliers['DFFITS'])
axes[1, 0].set_xlabel('Índice de observación')
axes[1, 0].set_ylabel('DFFITS')
axes[1, 0].set_title('DFFITS por observación')
# Añadir líneas de referencia
cutoff_dffits = 2 * np.sqrt((X_outliers.shape[1] + 1) / X_outliers.shape[0])
axes[1, 0].axhline(y=cutoff_dffits, color='r', linestyle='--')
axes[1, 0].axhline(y=-cutoff_dffits, color='r', linestyle='--')
# Marcar outliers conocidos
axes[1, 0].scatter(outliers_indices, influyentes_outliers.iloc[outliers_indices]['DFFITS'],
                  color='red', marker='x', s=100)

# Gráfico de detección de outliers
axes[1, 1].bar(influyentes_outliers.index, influyentes_outliers['Total Flags'])
axes[1, 1].set_xlabel('Índice de observación')
axes[1, 1].set_ylabel('Número de flags')
axes[1, 1].set_title('Total de flags por observación')
# Marcar outliers conocidos
axes[1, 1].scatter(outliers_indices, influyentes_outliers.iloc[outliers_indices]['Total Flags'],
                  color='red', marker='x', s=100)

plt.tight_layout()
plt.show()

# Gráfico de influencia utilizando statsmodels
fig, ax = plt.subplots(figsize=(10, 6))
influence_plot(modelo_outliers, ax=ax)
ax.set_title('Gráfico de Influencia (statsmodels)')
# Marcar outliers conocidos
for idx in outliers_indices:
    ax.annotate(f'{idx}', (influyentes_outliers['Leverage'][idx], 
                          influyentes_outliers['Residuos Estandarizados'][idx]),
               xytext=(5, 5), textcoords='offset points', color='red')
plt.tight_layout()
plt.show()


# Comparación de Métodos Robustos

En esta sección, compararemos diferentes métodos de regresión robusta y su capacidad para resistir el efecto de valores atípicos.


In [ ]:

# Comparar métodos robustos en el conjunto de datos sin outliers
print("Comparación de métodos robustos en datos sin outliers:")
resultados_clean = comparar_metodos_robustos(X_clean, y_clean)
print(resultados_clean)

# Comparar métodos robustos en el conjunto de datos con outliers
print("\nComparación de métodos robustos en datos con outliers:")
resultados_outliers = comparar_metodos_robustos(X_outliers, y_outliers, outliers_indices)
print(resultados_outliers)

# Cálculo de diferencias porcentuales entre coeficientes
print("\nDiferencia porcentual respecto a OLS para datos sin outliers:")
diff_clean = resultados_clean.copy()
for col in diff_clean.columns:
    diff_clean[col] = 100 * (diff_clean[col] - diff_clean.loc['OLS', col]) / abs(diff_clean.loc['OLS', col])
diff_clean = diff_clean.iloc[1:, :]  # Excluir OLS
print(diff_clean)

print("\nDiferencia porcentual respecto a OLS para datos con outliers:")
diff_outliers = resultados_outliers.copy()
for col in diff_outliers.columns:
    diff_outliers[col] = 100 * (diff_outliers[col] - diff_outliers.loc['OLS', col]) / abs(diff_outliers.loc['OLS', col])
diff_outliers = diff_outliers.iloc[1:, :]  # Excluir OLS
print(diff_outliers)

# Visualización de los coeficientes estimados
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Coeficientes para datos sin outliers
coef_names = resultados_clean.columns
for i, coef in enumerate(coef_names):
    axes[i].barh(resultados_clean.index, resultados_clean[coef])
    axes[i].axvline(x=resultados_clean.loc['OLS', coef], color='r', linestyle='--')
    axes[i].set_title(f'Estimaciones de {coef} (Sin outliers)')
    axes[i].set_xlabel('Valor del coeficiente')

plt.tight_layout()
plt.show()

# Visualización de los coeficientes estimados con outliers
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Coeficientes para datos con outliers
for i, coef in enumerate(coef_names):
    axes[i].barh(resultados_outliers.index, resultados_outliers[coef])
    axes[i].axvline(x=resultados_clean.loc['OLS', coef], color='g', linestyle='--', label='OLS sin outliers')
    axes[i].axvline(x=resultados_outliers.loc['OLS', coef], color='r', linestyle='--', label='OLS con outliers')
    axes[i].set_title(f'Estimaciones de {coef} (Con outliers)')
    axes[i].set_xlabel('Valor del coeficiente')
    if i == 0:
        axes[i].legend()

plt.tight_layout()
plt.show()

# Visualización de las rectas de regresión
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Para X1 vs Y
axes[0].scatter(X_outliers['X1'], y_outliers, alpha=0.6, label='Datos')
axes[0].scatter(X_outliers.iloc[outliers_indices]['X1'], y_outliers.iloc[outliers_indices], 
               color='red', marker='x', s=100, label='Outliers')

# Generar valores para graficar las rectas
x1_range = np.linspace(X_outliers['X1'].min(), X_outliers['X1'].max(), 100)
x2_media = X_outliers['X2'].mean()

# Graficar rectas para diferentes métodos
metodos = ['OLS', 'Huber', 'Tukey', 'LTS', 'RANSAC']
colores = ['black', 'blue', 'green', 'purple', 'orange']

for i, metodo in enumerate(metodos):
    if metodo in resultados_outliers.index:
        coefs = resultados_outliers.loc[metodo]
        y_pred = coefs['Intercepto'] + coefs['Beta_1'] * x1_range + coefs['Beta_2'] * x2_media
        axes[0].plot(x1_range, y_pred, color=colores[i], linewidth=2, label=metodo)

axes[0].set_title('X1 vs Y con diferentes modelos robustos')
axes[0].set_xlabel('X1')
axes[0].set_ylabel('Y')
axes[0].legend()

# Para X2 vs Y
axes[1].scatter(X_outliers['X2'], y_outliers, alpha=0.6, label='Datos')
axes[1].scatter(X_outliers.iloc[outliers_indices]['X2'], y_outliers.iloc[outliers_indices], 
               color='red', marker='x', s=100, label='Outliers')

# Generar valores para graficar las rectas
x2_range = np.linspace(X_outliers['X2'].min(), X_outliers['X2'].max(), 100)
x1_media = X_outliers['X1'].mean()

# Graficar rectas para diferentes métodos
for i, metodo in enumerate(metodos):
    if metodo in resultados_outliers.index:
        coefs = resultados_outliers.loc[metodo]
        y_pred = coefs['Intercepto'] + coefs['Beta_1'] * x1_media + coefs['Beta_2'] * x2_range
        axes[1].plot(x2_range, y_pred, color=colores[i], linewidth=2, label=metodo)

axes[1].set_title('X2 vs Y con diferentes modelos robustos')
axes[1].set_xlabel('X2')
axes[1].set_ylabel('Y')
axes[1].legend()

plt.tight_layout()
plt.show()

# Evaluación de la precisión predictiva
X_test, y_test, _ = generar_datos_lineales(n=50, p=2, outliers_prop=0.0, seed=123)
X_test_const = add_constant(X_test)

print("\nEvaluación de la precisión predictiva en datos de prueba sin outliers:")
errores = {}

for metodo in resultados_outliers.index:
    coefs = resultados_outliers.loc[metodo].values
    y_pred = X_test_const @ coefs
    mse = np.mean((y_test - y_pred) ** 2)
    mae = np.mean(np.abs(y_test - y_pred))
    errores[metodo] = {'MSE': mse, 'MAE': mae}

errores_df = pd.DataFrame(errores).T
print(errores_df)

# Visualización de errores de predicción
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

errores_df['MSE'].sort_values().plot(kind='bar', ax=axes[0])
axes[0].set_title('Error Cuadrático Medio (MSE)')
axes[0].set_ylabel('MSE')

errores_df['MAE'].sort_values().plot(kind='bar', ax=axes[1])
axes[1].set_title('Error Absoluto Medio (MAE)')
axes[1].set_ylabel('MAE')

plt.tight_layout()
plt.show()


# Análisis del Comportamiento de Estimadores Robustos

En esta sección, exploraremos cómo los diferentes estimadores robustos se comportan a medida que aumenta la proporción de outliers en los datos.


In [ ]:

def experimento_proporcion_outliers(n=100, p=2, proporciones=[0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3], 
                                   repeticiones=10, outliers_factor=10):
    """
    Experimento para evaluar el efecto de la proporción de outliers en diferentes métodos.

    Parámetros:
    -----------
    n : int
        Número de observaciones
    p : int
        Número de predictores
    proporciones : list
        Lista de proporciones de outliers a evaluar
    repeticiones : int
        Número de repeticiones por proporción
    outliers_factor : float
        Factor de contaminación para los outliers

    Retorna:
    --------
    resultados : DataFrame
        DataFrame con los errores promedio para cada método y proporción
    """
    # Generar coeficientes verdaderos fijos
    beta_true = np.concatenate(([2.0], np.random.uniform(-2, 2, p)))

    # Métodos a evaluar
    metodos = ['OLS', 'Huber', 'Tukey', 'RANSAC', 'LTS']

    # Inicializar resultados
    resultados = {}

    # Para cada proporción de outliers
    for prop in proporciones:
        print(f"Evaluando proporción de outliers: {prop}")

        # Inicializar errores por método
        errores_metodo = {metodo: [] for metodo in metodos}

        # Repetir experimento
        for rep in range(repeticiones):
            # Generar datos de entrenamiento con outliers
            X_train, y_train, _ = generar_datos_lineales(n=n, p=p, beta_true=beta_true, 
                                                       outliers_prop=prop, outliers_factor=outliers_factor,
                                                       seed=42 + rep)

            # Generar datos de prueba sin outliers
            X_test, y_test, _ = generar_datos_lineales(n=50, p=p, beta_true=beta_true, 
                                                     outliers_prop=0.0, seed=1000 + rep)
            X_test_const = add_constant(X_test)

            # Ajustar modelos y evaluar
            resultados_modelos = comparar_metodos_robustos(X_train, y_train)

            # Calcular errores de predicción
            for metodo in metodos:
                if metodo in resultados_modelos.index:
                    coefs = resultados_modelos.loc[metodo].values
                    y_pred = X_test_const @ coefs
                    mse = np.mean((y_test - y_pred) ** 2)
                    errores_metodo[metodo].append(mse)

        # Calcular promedios
        for metodo in metodos:
            if errores_metodo[metodo]:
                resultados.setdefault(metodo, []).append(np.mean(errores_metodo[metodo]))
            else:
                resultados.setdefault(metodo, []).append(np.nan)

    # Crear DataFrame
    resultados_df = pd.DataFrame(resultados, index=proporciones)
    resultados_df.index.name = 'Proporción de outliers'

    return resultados_df

# Ejecutar experimento con una configuración simple para demostración
# Reducimos repeticiones y proporciones para que sea más rápido
experimento_df = experimento_proporcion_outliers(n=100, p=2, 
                                               proporciones=[0, 0.05, 0.1, 0.15, 0.2], 
                                               repeticiones=2, outliers_factor=7)

print("Resultados del experimento:")
print(experimento_df)

# Normalizar resultados respecto a OLS para mejor visualización
experimento_normalizado = experimento_df.div(experimento_df['OLS'], axis=0)
print("\nResultados normalizados respecto a OLS:")
print(experimento_normalizado)

# Visualización de resultados
plt.figure(figsize=(12, 8))

# Gráfico de errores absolutos
plt.subplot(2, 1, 1)
for metodo in experimento_df.columns:
    plt.plot(experimento_df.index, experimento_df[metodo], marker='o', label=metodo)
plt.xlabel('Proporción de outliers')
plt.ylabel('MSE promedio')
plt.title('Error cuadrático medio por método y proporción de outliers')
plt.grid(True, alpha=0.3)
plt.legend()

# Gráfico de errores relativos a OLS
plt.subplot(2, 1, 2)
for metodo in experimento_normalizado.columns:
    if metodo != 'OLS':
        plt.plot(experimento_normalizado.index, experimento_normalizado[metodo], 
                marker='o', label=metodo)
plt.axhline(y=1, color='r', linestyle='--', label='OLS')
plt.xlabel('Proporción de outliers')
plt.ylabel('MSE relativo a OLS')
plt.title('Error relativo a OLS por método y proporción de outliers')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

# Análisis de la variabilidad de los coeficientes
print("\nAnálisis de la variabilidad de los coeficientes:")

def evaluar_variabilidad_coeficientes(n=100, p=2, outliers_prop=0.2, repeticiones=10, outliers_factor=7):
    """
    Evalúa la variabilidad de los coeficientes estimados por diferentes métodos.
    """
    # Generar coeficientes verdaderos fijos
    beta_true = np.concatenate(([2.0], np.random.uniform(-2, 2, p)))

    # Inicializar almacenamiento para coeficientes
    coeficientes = {}

    # Para cada repetición
    for rep in range(repeticiones):
        # Generar datos
        X, y, _ = generar_datos_lineales(n=n, p=p, beta_true=beta_true, 
                                       outliers_prop=outliers_prop, outliers_factor=outliers_factor,
                                       seed=42 + rep)

        # Ajustar modelos
        resultados = comparar_metodos_robustos(X, y)

        # Almacenar coeficientes
        for metodo in resultados.index:
            if metodo not in coeficientes:
                coeficientes[metodo] = {col: [] for col in resultados.columns}

            for col in resultados.columns:
                coeficientes[metodo][col].append(resultados.loc[metodo, col])

    # Calcular estadísticas
    estadisticas = {}
    for metodo, datos in coeficientes.items():
        estadisticas[metodo] = {}
        for coef, valores in datos.items():
            estadisticas[metodo][f"{coef}_media"] = np.mean(valores)
            estadisticas[metodo][f"{coef}_std"] = np.std(valores)
            estadisticas[metodo][f"{coef}_cv"] = np.std(valores) / abs(np.mean(valores)) * 100  # Coef. variación

    return pd.DataFrame(estadisticas).T, beta_true

# Evaluar variabilidad con menos repeticiones para demostración
variabilidad_df, beta_verdadero = evaluar_variabilidad_coeficientes(repeticiones=3)

print("Estadísticas de variabilidad de coeficientes:")
print(variabilidad_df)

print("\nCoeficientes verdaderos:")
for i, coef in enumerate(['Intercepto'] + [f'Beta_{j+1}' for j in range(len(beta_verdadero)-1)]):
    print(f"{coef}: {beta_verdadero[i]:.4f}")

# Visualización de la variabilidad
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Para cada coeficiente
for i, coef in enumerate(['Intercepto', 'Beta_1', 'Beta_2']):
    # Medias
    medias = [variabilidad_df.loc[metodo, f"{coef}_media"] for metodo in variabilidad_df.index]
    # Desviaciones estándar
    stds = [variabilidad_df.loc[metodo, f"{coef}_std"] for metodo in variabilidad_df.index]

    # Graficar
    axes[i].errorbar(range(len(variabilidad_df.index)), medias, yerr=stds, fmt='o', capsize=5)
    axes[i].set_xticks(range(len(variabilidad_df.index)))
    axes[i].set_xticklabels(variabilidad_df.index, rotation=45)
    axes[i].axhline(y=beta_verdadero[i], color='r', linestyle='--', label='Valor verdadero')
    axes[i].set_title(f'Estimaciones de {coef}')
    axes[i].set_ylabel('Valor del coeficiente')
    axes[i].legend()

plt.tight_layout()
plt.show()


# Ejercicios

A continuación se presentan varios ejercicios para practicar con los conceptos de valores influyentes y regresión robusta.

### Ejercicio 1
Genera un conjunto de datos simulado con las siguientes características:
- 100 observaciones
- 3 variables predictoras
- 15% de observaciones atípicas en X1 y Y

1. Identifica los valores influyentes utilizando las diferentes métricas implementadas.
2. Calcula el cambio en los coeficientes al excluir cada una de las observaciones más influyentes (calcular DFBETAS).
3. Visualiza el impacto de los valores influyentes en un gráfico apropiado.

### Ejercicio 2
Utiliza los datos generados en el Ejercicio 1 para:

1. Implementa tu propia función de peso para un M-estimador (diferente de Huber y Tukey).
2. Compara el desempeño de tu función con los métodos robustos implementados en el taller.
3. Evalúa la precisión predictiva de cada método en un conjunto de prueba sin valores atípicos.

### Ejercicio 3
Carga el conjunto de datos `diabetes` de sklearn:

```python
from sklearn.datasets import load_diabetes
diabetes = load_diabetes()
X_diabetes = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_diabetes = pd.Series(diabetes.target, name='Progression')
```

1. Ajusta un modelo de regresión lineal clásico (OLS).
2. Identifica valores influyentes.
3. Ajusta al menos tres modelos robustos diferentes.
4. Compara los coeficientes estimados y la precisión predictiva de los diferentes modelos.
5. ¿Cambian significativamente los coeficientes al usar métodos robustos? ¿Qué indica esto sobre la presencia de valores atípicos influyentes?

### Ejercicio 4
Implementa una función para detectar outliers multivariados utilizando la distancia de Mahalanobis:

```python
def distancia_mahalanobis(X):
    # Calcular media
    mu = np.mean(X, axis=0)

    # Calcular matriz de covarianza
    cov = np.cov(X, rowvar=False)

    # Calcular inversa
    cov_inv = np.linalg.inv(cov)

    # Calcular distancias
    diff = X - mu
    dist = np.sqrt(np.sum(diff @ cov_inv * diff, axis=1))

    return dist
```

1. Aplica esta función a los datos generados en el Ejercicio 1.
2. Compara los outliers detectados con los valores influyentes identificados mediante las métricas estudiadas.
3. ¿Es siempre un outlier multivariado un valor influyente en la regresión? Explica.

### Ejercicio 5
Crea un experimento para evaluar la "robustez del punto de ruptura" de diferentes métodos:

1. Genera conjuntos de datos con proporciones cada vez mayores de valores atípicos (de 0% a 50%).
2. Para cada conjunto, aplica diferentes métodos robustos y OLS.
3. Evalúa el punto donde cada método "se rompe" (cuando el error se dispara).
4. Visualiza los resultados en un gráfico.

Hint: Un estimador como OLS teóricamente tiene un punto de ruptura de 0%, mientras que LTS puede resistir hasta 50% de contaminación.


# Conclusiones

En este taller hemos explorado los conceptos de valores influyentes y regresión robusta, cubriendo desde los fundamentos teóricos hasta la implementación práctica de diferentes métodos. A continuación, se resumen los puntos clave:

1. **Valores atípicos e influyentes**:
   - Los valores atípicos (outliers) son observaciones con valores inusuales en X o Y.
   - Los valores influyentes son observaciones que afectan significativamente los parámetros estimados del modelo.
   - Es posible tener valores atípicos que no sean influyentes, y viceversa.

2. **Métricas de influencia**:
   - El leverage mide la influencia potencial basada solo en valores X.
   - Los residuos estandarizados y estudentizados evalúan la desviación en Y.
   - DFFITS mide el cambio en las predicciones cuando se excluye una observación.
   - DFBETAS mide el cambio en los coeficientes cuando se excluye una observación.
   - La distancia de Cook combina información de leverage y residuos para una evaluación integral.

3. **Métodos de regresión robusta**:
   - Los M-estimadores (como Huber y Tukey) utilizan funciones de pérdida robustas para reducir la influencia de valores atípicos.
   - Least Trimmed Squares (LTS) excluye directamente observaciones con residuos grandes.
   - RANSAC identifica y excluye outliers mediante un proceso iterativo de muestreo aleatorio.
   - Diferentes métodos robustos son más adecuados para diferentes situaciones, dependiendo del tipo y proporción de outliers.

4. **Comportamiento de los métodos robustos**:
   - OLS es altamente sensible a valores atípicos, con un punto de ruptura teórico de 0%.
   - Los métodos robustos tienen mayor estabilidad en presencia de outliers.
   - Existe un equilibrio entre robustez y eficiencia: los métodos más robustos pueden ser menos eficientes en datos limpios.
   - La proporción de outliers afecta críticamente el desempeño de cada método.

5. **Aplicaciones prácticas**:
   - La detección de valores influyentes es una etapa crucial en el análisis exploratorio de datos.
   - Los métodos robustos son especialmente útiles en situaciones donde no es factible inspeccionar y limpiar manualmente grandes conjuntos de datos.
   - La validación cruzada permite evaluar objetivamente el desempeño de diferentes métodos en datos específicos.

6. **Implementación computacional**:
   - Hemos implementado desde cero las principales métricas y métodos, lo que proporciona una comprensión profunda de los algoritmos.
   - Estas implementaciones sirven como base para el desarrollo de funciones personalizadas adaptadas a problemas específicos.
   - La comparación con implementaciones de bibliotecas como statsmodels y sklearn proporciona validación de nuestro trabajo.

La regresión robusta es una herramienta poderosa y necesaria en el análisis estadístico moderno, especialmente en situaciones con datos potencialmente contaminados. Comprender tanto los aspectos teóricos como los prácticos permite seleccionar y aplicar los métodos más apropiados para cada contexto.


# Referencias

1. Belsley, D. A., Kuh, E., & Welsch, R. E. (1980). *Regression Diagnostics: Identifying Influential Data and Sources of Collinearity*. John Wiley & Sons.

2. Cook, R. D. (1977). Detection of Influential Observation in Linear Regression. *Technometrics, 19*(1), 15-18.

3. Fox, J. (2016). *Applied Regression Analysis and Generalized Linear Models*. Sage Publications.

4. Hampel, F. R., Ronchetti, E. M., Rousseeuw, P. J., & Stahel, W. A. (2011). *Robust Statistics: The Approach Based on Influence Functions*. John Wiley & Sons.

5. Huber, P. J., & Ronchetti, E. M. (2009). *Robust Statistics* (2nd ed.). John Wiley & Sons.

6. Kutner, M. H., Nachtsheim, C. J., Neter, J., & Li, W. (2005). *Applied Linear Statistical Models* (5th ed.). McGraw-Hill/Irwin.

7. Rousseeuw, P. J., & Leroy, A. M. (2005). *Robust Regression and Outlier Detection*. John Wiley & Sons.

8. Rousseeuw, P. J. (1984). Least Median of Squares Regression. *Journal of the American Statistical Association, 79*(388), 871-880.

9. Wilcox, R. R. (2017). *Introduction to Robust Estimation and Hypothesis Testing* (4th ed.). Academic Press.

10. Yohai, V. J. (1987). High Breakdown-Point and High Efficiency Robust Estimates for Regression. *The Annals of Statistics, 15*(2), 642-656.


# Apéndice: Derivación Matemática de M-estimadores

En este apéndice, presentamos la derivación matemática de los M-estimadores utilizados en la regresión robusta.

## M-estimadores: Fundamentos

Los M-estimadores son una generalización del principio de máxima verosimilitud. Mientras que OLS minimiza la suma de errores cuadráticos:

$\min_{\beta} \sum_{i=1}^n (y_i - x_i^T\beta)^2$

Los M-estimadores minimizan una función objetivo diferente:

$\min_{\beta} \sum_{i=1}^n \rho(y_i - x_i^T\beta)$

donde $\rho$ es una función de pérdida robusta.

## Iteratively Reweighted Least Squares (IRLS)

Para encontrar el mínimo, derivamos e igualamos a cero:

$\sum_{i=1}^n \psi(y_i - x_i^T\beta)x_i = 0$

donde $\psi(e) = \rho'(e)$ es la derivada de $\rho$.

Definiendo la función de peso $w(e) = \psi(e)/e$, podemos reescribir como:

$\sum_{i=1}^n w(y_i - x_i^T\beta)(y_i - x_i^T\beta)x_i = 0$

Esto lleva al algoritmo IRLS:

1. Comenzar con un estimador inicial $\beta^{(0)}$ (generalmente OLS)
2. En cada iteración $k$:
   - Calcular residuos $e_i^{(k)} = y_i - x_i^T\beta^{(k)}$
   - Calcular pesos $w_i^{(k)} = w(e_i^{(k)})$
   - Actualizar $\beta^{(k+1)} = (X^TWX)^{-1}X^TWy$ donde $W = \text{diag}(w_i^{(k)})$
3. Repetir hasta convergencia

## Funciones de pérdida específicas

### Función de Huber

$\rho(e) = \begin{cases}
\frac{1}{2}e^2 & \text{si } |e| \leq k \\
k|e| - \frac{1}{2}k^2 & \text{si } |e| > k
\end{cases}$

La función de peso correspondiente es:

$w(e) = \begin{cases}
1 & \text{si } |e| \leq k \\
\frac{k}{|e|} & \text{si } |e| > k
\end{cases}$

### Función de Tukey (Biweight)

$\rho(e) = \begin{cases}
\frac{k^2}{6}\left[1-\left(1-\left(\frac{e}{k}\right)^2\right)^3\right] & \text{si } |e| \leq k \\
\frac{k^2}{6} & \text{si } |e| > k
\end{cases}$

La función de peso correspondiente es:

$w(e) = \begin{cases}
\left[1-\left(\frac{e}{k}\right)^2\right]^2 & \text{si } |e| \leq k \\
0 & \text{si } |e| > k
\end{cases}$

## Estimación de la escala

Para hacer que los M-estimadores sean invariantes a la escala, normalmente se estandarizan los residuos:

$\rho\left(\frac{y_i - x_i^T\beta}{\hat{\sigma}}\right)$

donde $\hat{\sigma}$ es un estimador robusto de escala. Una opción común es la Desviación Absoluta Mediana (MAD):

$\hat{\sigma} = \frac{\text{median}|e_i - \text{median}(e_i)|}{0.6745}$

El factor 0.6745 hace que la MAD sea un estimador consistente de $\sigma$ para errores normales.

## Propiedades de robustez

La robustez de un estimador se mide por su **función de influencia** y su **punto de ruptura**:

- La función de influencia describe el efecto de una observación sobre el estimador.
- El punto de ruptura es la proporción máxima de observaciones atípicas que un estimador puede manejar antes de producir resultados arbitrariamente grandes.

Para OLS, la función de influencia es ilimitada y el punto de ruptura es 0%.
Para M-estimadores con función de Huber, el punto de ruptura es aproximadamente $1/p$ donde $p$ es el número de parámetros.
Para LTS y otros estimadores de alto punto de ruptura, este puede llegar hasta 50%.
